# 03. Baseline Forecasting Models
### Energy Demand Forecasting Pipeline

This notebook implements and benchmarks two non-trivial baseline models:
1. **Naive Forecaster**: Repeats the last observed training value ($\hat{y}_{t+h} = y_T$).
2. **Seasonal-Naive Forecaster**: Repeats the observation from the corresponding day in the prior seasonal cycle ($\hat{y}_{t+h} = y_{T+h-s}$, where $s=7$ for daily data).

**Key Operations:**
- Fit baseline models on the 70% training split
- Predict across the entire 30% holdout test horizon
- Compute benchmark error metrics (MAE, RMSE, MAPE)

## 1. Setup & Load Processed Splits

In [2]:
import os
import sys
import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

try:
    from IPython.display import display
except ImportError:
    display = print

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("energy_forecasting")

# Robust directory discovery
for base in [Path("."), Path(".."), Path("../..")]:
    candidate = base / "ml" / "data"
    if candidate.exists():
        PROJECT_ROOT = base.resolve()
        break
else:
    PROJECT_ROOT = Path(".").resolve()

ML_DIR = PROJECT_ROOT / "ml"
DATA_DIR = ML_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
ARTIFACTS_DIR = ML_DIR / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
FORECASTS_DIR = ARTIFACTS_DIR / "forecasts"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

for d in [PROCESSED_DIR, REPORTS_DIR, MODELS_DIR, FORECASTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Locate raw 60min singleindex dataset
RAW_DATA_PATH = None
for candidate in [
    DATA_DIR / "opsd-time_series-2020-10-06" / "opsd-time_series-2020-10-06" / "time_series_60min_singleindex.csv",
    DATA_DIR / "time_series_60min_singleindex.csv",
    Path("ml/data/opsd-time_series-2020-10-06/opsd-time_series-2020-10-06/time_series_60min_singleindex.csv"),
    Path("data/opsd-time_series-2020-10-06/opsd-time_series-2020-10-06/time_series_60min_singleindex.csv"),
]:
    if candidate.exists():
        RAW_DATA_PATH = candidate.resolve()
        break

TIMESTAMP_COL = "utc_timestamp"
TARGET_COL_RAW = "DE_load_actual_entsoe_transparency"
TARGET_UNIT = "MW"
DS_COL = "ds"
Y_COL = "y"
TRAIN_RATIO = 0.70

print(f"Project root  : {PROJECT_ROOT}")
print(f"Raw data path : {RAW_DATA_PATH}")

train_daily = pd.read_csv(PROCESSED_DIR / "train_daily.csv", parse_dates=[DS_COL])
test_daily = pd.read_csv(PROCESSED_DIR / "test_daily.csv", parse_dates=[DS_COL])
print(f"Loaded train_daily ({len(train_daily):,} rows) and test_daily ({len(test_daily):,} rows).")

Project root  : C:\Users\Soham\OneDrive\Desktop\Energy Demand Forecasting
Raw data path : C:\Users\Soham\OneDrive\Desktop\Energy Demand Forecasting\ml\data\opsd-time_series-2020-10-06\opsd-time_series-2020-10-06\time_series_60min_singleindex.csv
Loaded train_daily (1,470 rows) and test_daily (631 rows).


## 2. Baseline Models Implementation & Benchmark Evaluation

In [3]:
# Baseline Forecasters Implementation
class NaiveForecaster:
    """Repeats the last observed training value across the forecast horizon."""
    def __init__(self):
        self.last_value = None
        
    def fit(self, train_df: pd.DataFrame):
        self.last_value = float(train_df[Y_COL].iloc[-1])
        return self
        
    def predict(self, test_df: pd.DataFrame) -> pd.DataFrame:
        preds = np.full(len(test_df), self.last_value)
        return pd.DataFrame({DS_COL: test_df[DS_COL].values, "yhat": preds})

class SeasonalNaiveForecaster:
    """Repeats the observed values from the most recent full seasonal cycle (period=s)."""
    def __init__(self, period: int = 7):
        self.period = period
        self.train_values = None
        
    def fit(self, train_df: pd.DataFrame):
        self.train_values = train_df[Y_COL].values.copy()
        return self
        
    def predict(self, test_df: pd.DataFrame) -> pd.DataFrame:
        n_train = len(self.train_values)
        horizon = len(test_df)
        preds = np.array([
            self.train_values[n_train - self.period + (i % self.period)]
            for i in range(horizon)
        ])
        return pd.DataFrame({DS_COL: test_df[DS_COL].values, "yhat": preds})

# Evaluation metrics helper
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2)}

# Fit & Evaluate Daily Baselines
naive_daily = NaiveForecaster().fit(train_daily)
pred_naive_daily = naive_daily.predict(test_daily)

s_naive_daily = SeasonalNaiveForecaster(period=7).fit(train_daily)
pred_s_naive_daily = s_naive_daily.predict(test_daily)

# Save baseline predictions
pred_naive_daily.to_csv(FORECASTS_DIR / "naive_predictions_daily.csv", index=False)
pred_s_naive_daily.to_csv(FORECASTS_DIR / "seasonal_naive_predictions_daily.csv", index=False)

metrics_naive = compute_metrics(test_daily['y'], pred_naive_daily['yhat'])
metrics_s_naive = compute_metrics(test_daily['y'], pred_s_naive_daily['yhat'])

baseline_comparison = pd.DataFrame([
    {"Model": "Naive (Last Value)", **metrics_naive},
    {"Model": "Seasonal Naive (7-Day Lag)", **metrics_s_naive}
])
print("=== BASELINE BENCHMARKS ON HOLDOUT TEST SET ===")
baseline_comparison

=== BASELINE BENCHMARKS ON HOLDOUT TEST SET ===


,Model,MAE,RMSE,MAPE (%)
0,Naive (Last Value),11672.51,13479.45,23.32
1,Seasonal Naive (7-Day Lag),5501.56,6785.79,10.68
